part 1: preparing the data, one-hot encoding

In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

bank = fetch_ucirepo(id=222)
X = bank.data.features.drop(columns=["duration"])
y = (bank.data.targets["y"] == "yes").astype(int)

X = pd.get_dummies(X)
print(X.shape)
print(X.columns.tolist())

(45211, 46)
['age', 'balance', 'day_of_week', 'campaign', 'pdays', 'previous', 'job_admin.', 'job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed', 'marital_divorced', 'marital_married', 'marital_single', 'education_primary', 'education_secondary', 'education_tertiary', 'default_no', 'default_yes', 'housing_no', 'housing_yes', 'loan_no', 'loan_yes', 'contact_cellular', 'contact_telephone', 'month_apr', 'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'poutcome_failure', 'poutcome_other', 'poutcome_success']


part 2: Splitting into train and test

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(36168, 46) (9043, 46)
0.11698186241981863 0.11699657193409267


part 3: Training the model

In [3]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

jobs = bank.data.features.loc[X_test.index, "job"]

for i in range(10):
    age = X_test["age"].iloc[i]
    job = jobs.iloc[i]
    balance = X_test["balance"].iloc[i]
    chance = probs[i] * 100
    real = "yes" if y_test.iloc[i] == 1 else "no"
    print(f"Customer {i+1}: {age}-year-old {job}, balance €{balance:,}. Model says {chance:.0f}% chance of yes. Really said: {real}.")

Customer 1: 40-year-old blue-collar, balance €640. Model says 1% chance of yes. Really said: no.
Customer 2: 44-year-old technician, balance €378. Model says 3% chance of yes. Really said: no.
Customer 3: 31-year-old services, balance €356. Model says 3% chance of yes. Really said: no.
Customer 4: 36-year-old blue-collar, balance €655. Model says 2% chance of yes. Really said: no.
Customer 5: 34-year-old services, balance €1,921. Model says 12% chance of yes. Really said: no.
Customer 6: 41-year-old management, balance €1,138. Model says 5% chance of yes. Really said: no.
Customer 7: 43-year-old entrepreneur, balance €2,589. Model says 9% chance of yes. Really said: no.
Customer 8: 64-year-old retired, balance €109. Model says 80% chance of yes. Really said: yes.
Customer 9: 35-year-old admin., balance €1,803. Model says 16% chance of yes. Really said: no.
Customer 10: 44-year-old services, balance €2,279. Model says 8% chance of yes. Really said: no.


c:\Users\USER\OneDrive\Desktop\bank-recommendation-engine\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Lesson 4 – Is the model any good?

part 1: Scaling (fixing the warning)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
probs = model.predict_proba(X_test_scaled)[:, 1]

In [5]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

jobs = bank.data.features.loc[X_test.index, "job"]

for i in range(10):
    age = X_test["age"].iloc[i]
    job = jobs.iloc[i]
    balance = X_test["balance"].iloc[i]
    chance = probs[i] * 100
    real = "yes" if y_test.iloc[i] == 1 else "no"
    print(f"Customer {i+1}: {age}-year-old {job}, balance €{balance:,}. Model says {chance:.0f}% chance of yes. Really said: {real}.")

Customer 1: 40-year-old blue-collar, balance €640. Model says 1% chance of yes. Really said: no.
Customer 2: 44-year-old technician, balance €378. Model says 3% chance of yes. Really said: no.
Customer 3: 31-year-old services, balance €356. Model says 3% chance of yes. Really said: no.
Customer 4: 36-year-old blue-collar, balance €655. Model says 2% chance of yes. Really said: no.
Customer 5: 34-year-old services, balance €1,921. Model says 12% chance of yes. Really said: no.
Customer 6: 41-year-old management, balance €1,138. Model says 5% chance of yes. Really said: no.
Customer 7: 43-year-old entrepreneur, balance €2,589. Model says 9% chance of yes. Really said: no.
Customer 8: 64-year-old retired, balance €109. Model says 80% chance of yes. Really said: yes.
Customer 9: 35-year-old admin., balance €1,803. Model says 16% chance of yes. Really said: no.
Customer 10: 44-year-old services, balance €2,279. Model says 8% chance of yes. Really said: no.


c:\Users\USER\OneDrive\Desktop\bank-recommendation-engine\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Cell 1: Load the data
Downloads the bank data, removes the leaking column, and turns yes/no into 1/0.

In [6]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

bank = fetch_ucirepo(id=222)
X = bank.data.features.drop(columns=["duration"])
y = (bank.data.targets["y"] == "yes").astype(int)

Cell 2: One-hot encode
Turns text columns into 1/0 columns so the model can read them (15 → 46 columns).

In [7]:
X = pd.get_dummies(X)
print(X.shape)

(45211, 46)


Cell 3: Split into train and test
Keeps 20% of customers hidden as the exam, with the same yes-rate in both parts.

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(36168, 46) (9043, 46)


Cell 4: Scale and train
Converts every column to the same "steps from average" units, then trains the model.

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
probs = model.predict_proba(X_test_scaled)[:, 1]

Cell 5: Print readable results
Shows the first 10 test customers as sentences, using the original (unscaled) age and balance.

In [10]:
jobs = bank.data.features.loc[X_test.index, "job"]

for i in range(10):
    age = X_test["age"].iloc[i]
    job = jobs.iloc[i]
    balance = X_test["balance"].iloc[i]
    chance = probs[i] * 100
    real = "yes" if y_test.iloc[i] == 1 else "no"
    print(f"Customer {i+1}: {age}-year-old {job}, balance €{balance:,}. Model says {chance:.0f}% chance of yes. Really said: {real}.")

Customer 1: 40-year-old blue-collar, balance €640. Model says 1% chance of yes. Really said: no.
Customer 2: 44-year-old technician, balance €378. Model says 3% chance of yes. Really said: no.
Customer 3: 31-year-old services, balance €356. Model says 4% chance of yes. Really said: no.
Customer 4: 36-year-old blue-collar, balance €655. Model says 2% chance of yes. Really said: no.
Customer 5: 34-year-old services, balance €1,921. Model says 13% chance of yes. Really said: no.
Customer 6: 41-year-old management, balance €1,138. Model says 5% chance of yes. Really said: no.
Customer 7: 43-year-old entrepreneur, balance €2,589. Model says 7% chance of yes. Really said: no.
Customer 8: 64-year-old retired, balance €109. Model says 84% chance of yes. Really said: yes.
Customer 9: 35-year-old admin., balance €1,803. Model says 11% chance of yes. Really said: no.
Customer 10: 44-year-old services, balance €2,279. Model says 10% chance of yes. Really said: no.


Cell 6: Accuracy and confusion matrix (part 2)
Checks the model against the "always say no" baseline and counts the types of mistakes.

In [11]:
from sklearn.metrics import accuracy_score, confusion_matrix

predictions = (probs >= 0.5).astype(int)

print("Model accuracy:", accuracy_score(y_test, predictions))
print("Always-no accuracy:", accuracy_score(y_test, [0] * len(y_test)))
print(confusion_matrix(y_test, predictions))

Model accuracy: 0.8936193741015149
Always-no accuracy: 0.8830034280659074
[[7891   94]
 [ 868  190]]


part 3: Moving the cut-off

In [12]:
for cutoff in [0.5, 0.3, 0.2]:
    predictions = (probs >= cutoff).astype(int)
    caught = ((predictions == 1) & (y_test == 1)).sum()
    missed = ((predictions == 0) & (y_test == 1)).sum()
    wasted = ((predictions == 1) & (y_test == 0)).sum()
    offers = caught + wasted
    print(f"Cut-off {cutoff:.0%}: {offers} offers, caught {caught} buyers, missed {missed}, wasted {wasted} offers")

Cut-off 50%: 284 offers, caught 190 buyers, missed 868, wasted 94 offers
Cut-off 30%: 634 offers, caught 351 buyers, missed 707, wasted 283 offers
Cut-off 20%: 999 offers, caught 466 buyers, missed 592, wasted 533 offers


In [13]:
for cutoff in [0.5, 0.3, 0.2, 0.1, 0.05]:
    predictions = (probs >= cutoff).astype(int)
    caught = ((predictions == 1) & (y_test == 1)).sum()
    missed = ((predictions == 0) & (y_test == 1)).sum()
    wasted = ((predictions == 1) & (y_test == 0)).sum()
    offers = caught + wasted
    print(f"Cut-off {cutoff:.0%}: {offers} offers, caught {caught} buyers, missed {missed}, wasted {wasted} offers")

Cut-off 50%: 284 offers, caught 190 buyers, missed 868, wasted 94 offers
Cut-off 30%: 634 offers, caught 351 buyers, missed 707, wasted 283 offers
Cut-off 20%: 999 offers, caught 466 buyers, missed 592, wasted 533 offers
Cut-off 10%: 3455 offers, caught 760 buyers, missed 298, wasted 2695 offers
Cut-off 5%: 6648 offers, caught 982 buyers, missed 76, wasted 5666 offers


### Lesson 4 findings
- Scaling (StandardScaler) puts every column in "steps from average" units. It fixed the ConvergenceWarning without changing results much.
- Accuracy is misleading: model 89.4% vs 88.3% for always saying "no".
- Confusion matrix at a 50% cut-off: 190 buyers caught, 868 missed, 94 wasted offers, 7,891 correctly ignored.
- Precision = of the people we offered to, how many said yes. Recall = of all real buyers, how many we found.
  - Recall starts from the buyers: how many did we reach?
  - Precision starts from the offers: how many hit?

| Cut-off | Offers | Caught | Recall | Precision |
|---|---|---|---|---|
| 50% | 284 | 190 | 18% | 67% |
| 30% | 634 | 351 | 33% | 55% |
| 20% | 999 | 466 | 44% | 47% |
| 10% | 3,455 | 760 | 72% | 22% |
| 5% | 6,648 | 982 | 93% | 15% |

- Lower cut-off = more buyers caught but more wasted offers, and each extra buyer costs more.
- For an app notification I'd pick 20%: cheap offers, but avoid annoying customers.
- Moving the cut-off only shifts the trade-off. To catch more buyers with the same number of offers, the model itself must improve.
- Benchmark for Lesson 5: with ~1,000 offers, this model catches 466 buyers.